In [ ]:
import pandas as pd
import json
import os
from tqdm.notebook import tqdm
from pydantic import RootModel
import argparse
from typing import List
import ollama 
import time
import pandas as pd
import csv
import re
# from sklearn.metrics import classification_report

In [ ]:
os.getcwd()

# go one up
os.chdir('..')
os.getcwd()

In [ ]:
from actor_annotator import ActorAnnotator, expand_actors_to_rows

In [ ]:
# ANNOTATED DATA FOR PROMPT TESTING
df = pd.read_csv('/home/eklk/data/storage_100gb/election_news/data/uk_annotated_test.csv', encoding='utf-8-sig', sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(df.shape)
df['article_id'] = df['article_id'].astype(int)
# Rename Text to text
df.rename(columns={'Text': 'text'}, inplace=True)
# merge article_title and text for text
df['text'] = df['article_title'] + '\n' + df['text']
df.head()

In [ ]:
df = df[df['main_topic'] == 1]
print(df.shape)

In [ ]:

# Initialize annotator with preferred model
annotator = ActorAnnotator(model_name="gpt-oss:20b", seed=42)
print("Actor annotator initialized successfully!")

In [ ]:
# Process the test sample
df_annotated = annotator.process_dataframe(
    df=df, 
    text_column='text', 
    id_column='article_id',
    title_column='article_title',
)
print("Test processing completed!")

In [ ]:
print(df_annotated[['article_id', 'news_actors']].head())

In [ ]:
df_annotated_copy = df_annotated.copy()

In [ ]:
# Expand test results to actor-level
df_annotated_copy['news_actors'] = df_annotated_copy['news_actors'].apply(annotator.clean_and_parse_actors)
df_annotated_copy.head()

In [ ]:
# expand_actors_to_rows
actors_df = expand_actors_to_rows(df_annotated_copy, id_column='article_id')
print(f"Test results: {len(actors_df)} actors found from {len(df_annotated)} articles in UK")
actors_df[['article_id', 'actor_name', 'actor_function', 'actor_pp']].head()

In [ ]:
# write annotated dataframe to csv
actors_df.to_csv('/home/eklk/data/storage_100gb/election_news/actors/test_data/uk_annotated_test_gptoss20b.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)